# データ拡張（Augmentation）自動生成スクリプト

In [ ]:
import os
import cv2
import albumentations as A
from tqdm import tqdm

# --- フォルダ設定 ---
IMG_DIR = "./data_train/v3/images/train"
LABEL_DIR = "./data_train/v3/labels/train"
OUT_IMG_DIR = "./data_train/v3/images/train"
OUT_LABEL_DIR = "./data_train/v3/labels/train"

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_LABEL_DIR, exist_ok=True)

# --- Augmentation設定 ---
transform = A.Compose([
    A.RandomBrightnessContrast(p=0.6),   # 明るさとコントラストのランダム調整
    A.HueSaturationValue(p=0.6),         # 色相・彩度・明度の変化
    A.Blur(blur_limit=3, p=0.3),         # ぼかし効果
    A.HorizontalFlip(p=0.5),             # 左右反転
    A.Affine(rotate=(-10, 10), scale=(0.9, 1.1), p=0.6),  # 回転とスケール変換
    A.Resize(640, 640)                   # YOLOでは画像サイズを統一する必要あり
],
    bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'])
)

# --- YOLOラベルを読み込む関数 ---
def read_yolo_label(label_path):
    boxes = []
    class_labels = []
    if not os.path.exists(label_path):
        return boxes, class_labels
    with open(label_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])
            boxes.append([x, y, w, h])
            class_labels.append(cls)
    return boxes, class_labels

# --- YOLOラベルを書き出す関数 ---
def save_yolo_label(path, boxes, class_labels):
    with open(path, "w") as f:
        for cls, (x, y, w, h) in zip(class_labels, boxes):
            f.write(f"{int(cls)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

# --- データ拡張の実行 ---
for img_name in tqdm(os.listdir(IMG_DIR)):
    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(IMG_DIR, img_name)
    label_path = os.path.join(LABEL_DIR, img_name.replace(".jpg", ".txt"))

    image = cv2.imread(img_path)
    bboxes, class_labels = read_yolo_label(label_path)

    # 各画像に対して複数のAugmentバージョンを作成
    for i in range(30):  # 各画像を30倍に増やす
        transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
        aug_img = transformed["image"]
        aug_bboxes = transformed["bboxes"]
        aug_labels = transformed["class_labels"]

        # バウンディングボックスがなくなった場合はスキップ
        if len(aug_bboxes) == 0:
            continue

        out_img_name = img_name.replace(".jpg", f"_aug_{i:03d}.jpg")
        out_label_name = img_name.replace(".jpg", f"_aug_{i:03d}.txt")

        cv2.imwrite(os.path.join(OUT_IMG_DIR, out_img_name), aug_img)
        save_yolo_label(os.path.join(OUT_LABEL_DIR, out_label_name), aug_bboxes, aug_labels)


# TRAIN

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # 事前学習モデルを使用

model.train(
    data="data_train/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=0.002,
    pretrained=True,
)


WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\LSI\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.3.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.214  Python-3.13.1 torch-2.8.0+cpu CPU (Intel Core(TM) i5-7300U 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_train/v1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynam